### Bước 1 — Copy code sang vùng ghi được, vá lỗi `read_jsonl`, cài thư viện

`/kaggle/input/...` là vùng CHỈ ĐỌC. Copy thư mục `research/` sang `/kaggle/working/repo` (ghi được), rồi vá lỗi `read_jsonl` dùng `str.splitlines()` sai (bị cắt nhầm dòng có ký tự `\x85`). Lỗi này đã sửa trong repo gốc, nhưng dataset code trên Kaggle là bản cũ nên chưa có bản vá.

In [4]:
import glob, shutil, os

hits = glob.glob('/kaggle/input/**/requirements-research.txt', recursive=True)
assert hits, 'Không tìm thấy requirements-research.txt — kiểm tra lại đã gắn đúng dataset code chưa'
RESEARCH_SRC = os.path.dirname(hits[0])
print('Tìm thấy thư mục research/ tại:', RESEARCH_SRC)

os.makedirs('/kaggle/working/repo', exist_ok=True)
shutil.copytree(RESEARCH_SRC, '/kaggle/working/repo/research', dirs_exist_ok=True)

!pip install -q -r /kaggle/working/repo/research/requirements-research.txt

# Vá lỗi read_jsonl (giống bản đã vá trên Colab)
path = "/kaggle/working/repo/research/offline_retrieval/common.py"
src = open(path, encoding="utf-8").read()
old = 'for line_number, raw_line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):'
new = 'for line_number, raw_line in enumerate(path.read_text(encoding="utf-8").split("\\n"), 1):'
assert src.count(old) == 1, f"tìm thấy {src.count(old)} lần, cần đúng 1"
open(path, "w", encoding="utf-8").write(src.replace(old, new))
print("Đã vá xong read_jsonl")

Tìm thấy thư mục research/ tại: /kaggle/input/datasets/yenkim1/kltn-pan-pc11-archive-raw/research_for_colab/research
Đã vá xong read_jsonl


### Bước 2 — Thay `bm25.py` bằng bản nhanh

**Vì sao phải thay:** bản BM25 cũ chạy bằng vòng lặp Python thuần, và không loại từ phổ biến ("the", "of"...) nên gần như đoạn nào cũng phải tính. Với 600.890 cửa sổ × 2.606 câu hỏi, nó chạy hơn 6 giờ vẫn chưa xong.

**Bản mới** tính bằng ma trận thưa (`scipy.sparse`): cùng công thức BM25 (k1=1,2; b=0,75), cùng cách gộp về tài liệu, cùng quy tắc xử lý hoà điểm. Đã kiểm tra trên dữ liệu ngẫu nhiên: thứ tự kết quả giống hệt bản cũ, điểm chỉ lệch cỡ 1e-12.

Bản cũ vẫn được giữ lại dưới tên `rank_bm25_reference` để đối chiếu khi cần.

*Ô dưới ghi đè file `bm25.py` trong `/kaggle/working/repo`, không đụng vào dataset gốc (chỉ đọc).*

In [5]:
%%writefile /kaggle/working/repo/research/offline_retrieval/bm25.py
"""Small deterministic Okapi BM25 baseline with no service dependency."""

from __future__ import annotations

from collections import Counter, defaultdict
import math
from typing import Any, Iterable

from .aggregation import render_ranked_candidates
from .common import tokenize


def rank_bm25_reference(
    corpus: Iterable[dict[str, Any]],
    queries: Iterable[dict[str, Any]],
    *,
    top_k: int,
    k1: float = 1.2,
    b: float = 0.75,
    group_by_document: bool = False,
) -> dict[str, list[dict[str, Any]]]:
    if top_k <= 0:
        raise ValueError("top_k must be positive")
    if k1 <= 0 or not 0 <= b <= 1:
        raise ValueError("BM25 requires k1 > 0 and b in [0, 1]")

    items = list(corpus)
    if not items:
        return {str(query["query_id"]): [] for query in queries}

    document_lengths: list[int] = []
    postings: dict[str, list[tuple[int, int]]] = defaultdict(list)
    seen_ids: set[str] = set()
    for item in items:
        chunk_id = item.get("chunk_id")
        text = item.get("text")
        if not isinstance(chunk_id, str) or not chunk_id:
            raise ValueError("Every corpus item requires a non-empty chunk_id")
        if chunk_id in seen_ids:
            raise ValueError(f"Duplicate chunk_id: {chunk_id}")
        seen_ids.add(chunk_id)
        if not isinstance(text, str):
            raise ValueError(f"Corpus item {chunk_id} requires string text")
        tokens = tokenize(text)
        counts = Counter(tokens)
        document_lengths.append(len(tokens))
        for term, frequency in counts.items():
            postings[term].append((len(document_lengths) - 1, frequency))

    average_length = sum(document_lengths) / len(items)
    average_length = average_length or 1.0
    corpus_size = len(items)

    results: dict[str, list[dict[str, Any]]] = {}
    seen_query_ids: set[str] = set()
    for query in queries:
        query_id = query.get("query_id")
        query_text = query.get("text")
        if not isinstance(query_id, str) or not query_id:
            raise ValueError("Every query requires a non-empty query_id")
        if query_id in seen_query_ids:
            raise ValueError(f"Duplicate query_id: {query_id}")
        seen_query_ids.add(query_id)
        if not isinstance(query_text, str):
            raise ValueError(f"Query {query_id} requires string text")

        query_terms = Counter(tokenize(query_text))
        scores: dict[int, float] = defaultdict(float)
        for term, query_frequency in query_terms.items():
            term_postings = postings.get(term)
            if not term_postings:
                continue
            document_frequency = len(term_postings)
            inverse_document_frequency = math.log(
                1
                + (corpus_size - document_frequency + 0.5)
                / (document_frequency + 0.5)
            )
            for item_index, frequency in term_postings:
                document_length = document_lengths[item_index]
                denominator = frequency + k1 * (
                    1 - b + b * document_length / average_length
                )
                scores[item_index] += query_frequency * inverse_document_frequency * (
                    frequency * (k1 + 1) / denominator
                )

        # A non-match has no retrieval evidence. Omitting zero-score chunks also
        # avoids arbitrary chunk-id tie breaks filling a requested top-k.
        scored = [
            (score, items[item_index]["chunk_id"], items[item_index])
            for item_index, score in scores.items()
            if score > 0.0
        ]

        results[query_id] = render_ranked_candidates(
            scored,
            top_k=top_k,
            group_by_document=group_by_document,
        )
    return results


# ---------------------------------------------------------------------------
# Bản nhanh (vector hoá bằng scipy.sparse). Cùng công thức, cùng cách gộp về
# tài liệu và cùng quy tắc phá hoà như rank_bm25_reference; chỉ khác thứ tự
# cộng số thực nên điểm có thể lệch ~1e-12.
# ---------------------------------------------------------------------------
def rank_bm25(
    corpus: Iterable[dict[str, Any]],
    queries: Iterable[dict[str, Any]],
    *,
    top_k: int,
    k1: float = 1.2,
    b: float = 0.75,
    group_by_document: bool = False,
    query_batch_size: int = 32,
) -> dict[str, list[dict[str, Any]]]:
    if not group_by_document:
        return rank_bm25_reference(
            corpus, queries, top_k=top_k, k1=k1, b=b, group_by_document=False
        )
    import numpy as np
    from array import array
    from scipy import sparse

    if top_k <= 0:
        raise ValueError("top_k must be positive")
    if k1 <= 0 or not 0 <= b <= 1:
        raise ValueError("BM25 requires k1 > 0 and b in [0, 1]")
    items = list(corpus)
    queries = list(queries)
    if not items:
        return {str(query["query_id"]): [] for query in queries}

    vocab: dict[str, int] = {}
    indptr = array("q", [0])
    indices = array("i")
    term_freqs = array("d")
    lengths = array("d")
    seen_ids: set[str] = set()
    for item in items:
        chunk_id = item.get("chunk_id")
        text = item.get("text")
        if not isinstance(chunk_id, str) or not chunk_id:
            raise ValueError("Every corpus item requires a non-empty chunk_id")
        if chunk_id in seen_ids:
            raise ValueError(f"Duplicate chunk_id: {chunk_id}")
        seen_ids.add(chunk_id)
        if not isinstance(text, str):
            raise ValueError(f"Corpus item {chunk_id} requires string text")
        document_id = item.get("document_id")
        if not isinstance(document_id, str) or not document_id:
            raise ValueError("Every retrieval candidate requires document_id")
        tokens = tokenize(text)
        lengths.append(len(tokens))
        for term, frequency in Counter(tokens).items():
            term_index = vocab.get(term)
            if term_index is None:
                term_index = vocab[term] = len(vocab)
            indices.append(term_index)
            term_freqs.append(frequency)
        indptr.append(len(indices))
    del seen_ids

    n_items, n_terms = len(items), len(vocab)
    indptr_np = np.frombuffer(indptr, dtype=np.int64)
    indices_np = np.frombuffer(indices, dtype=np.int32)
    tf = np.frombuffer(term_freqs, dtype=np.float64)
    lengths_np = np.frombuffer(lengths, dtype=np.float64)
    average_length = float(lengths_np.sum()) / n_items or 1.0
    df = np.bincount(indices_np, minlength=n_terms).astype(np.float64)
    idf = np.log(1 + (n_items - df + 0.5) / (df + 0.5))
    row_lengths = np.repeat(lengths_np, np.diff(indptr_np))
    weights = idf[indices_np] * (
        tf * (k1 + 1) / (tf + k1 * (1 - b + b * row_lengths / average_length))
    )
    del row_lengths, term_freqs
    # term x item: mỗi hàng là posting list đã nhân sẵn trọng số BM25
    term_item = sparse.csr_matrix(
        (weights, indices_np, indptr_np), shape=(n_items, n_terms)
    ).T.tocsr()
    del weights

    # Gom chunk theo tài liệu; trong mỗi tài liệu xếp chunk theo chunk_id để
    # argmax lấy chunk có id nhỏ nhất khi hoà (giống bản gốc).
    doc_ids = sorted({item["document_id"] for item in items})
    doc_pos = {d: i for i, d in enumerate(doc_ids)}
    item_doc = np.fromiter(
        (doc_pos[item["document_id"]] for item in items), dtype=np.int64, count=n_items
    )
    per_doc: list[list[int]] = [[] for _ in doc_ids]
    for item_index, item in enumerate(items):
        per_doc[item_doc[item_index]].append(item_index)
    per_doc_np = [
        np.array(sorted(members, key=lambda i: items[i]["chunk_id"]), dtype=np.int64)
        for members in per_doc
    ]
    del per_doc

    results: dict[str, list[dict[str, Any]]] = {}
    seen_query_ids: set[str] = set()
    for start in range(0, len(queries), query_batch_size):
        batch = queries[start : start + query_batch_size]
        rows, cols, vals, batch_ids = [], [], [], []
        for row, query in enumerate(batch):
            query_id = query.get("query_id")
            query_text = query.get("text")
            if not isinstance(query_id, str) or not query_id:
                raise ValueError("Every query requires a non-empty query_id")
            if query_id in seen_query_ids:
                raise ValueError(f"Duplicate query_id: {query_id}")
            seen_query_ids.add(query_id)
            if not isinstance(query_text, str):
                raise ValueError(f"Query {query_id} requires string text")
            batch_ids.append(query_id)
            for term, query_frequency in Counter(tokenize(query_text)).items():
                term_index = vocab.get(term)
                if term_index is not None:
                    rows.append(row)
                    cols.append(term_index)
                    vals.append(float(query_frequency))
        query_matrix = sparse.csr_matrix(
            (vals, (rows, cols)), shape=(len(batch), n_terms)
        )
        batch_scores = (query_matrix @ term_item).tocsr()
        for row, query_id in enumerate(batch_ids):
            lo, hi = batch_scores.indptr[row], batch_scores.indptr[row + 1]
            matched = batch_scores.indices[lo:hi]
            matched_scores = batch_scores.data[lo:hi]
            keep = matched_scores > 0.0
            matched, matched_scores = matched[keep], matched_scores[keep]
            if matched.size == 0:
                results[query_id] = []
                continue
            dense = np.zeros(n_items, dtype=np.float64)
            dense[matched] = matched_scores
            best = np.full(len(doc_ids), -np.inf)
            np.maximum.at(best, item_doc[matched], matched_scores)
            candidates = np.flatnonzero(best > 0.0)
            # (-điểm, document_id): doc_ids đã sort nên chỉ số = thứ tự theo id
            order = candidates[np.lexsort((candidates, -best[candidates]))][:top_k]
            ranked = []
            for rank, doc_index in enumerate(order, 1):
                members = per_doc_np[doc_index]
                item = items[int(members[int(np.argmax(dense[members]))])]
                ranked.append(
                    {
                        "document_id": item["document_id"],
                        "supporting_chunk_id": item["chunk_id"],
                        "score": float(best[doc_index]),
                        "rank": rank,
                        "source_spans": item.get("source_spans", []),
                    }
                )
            results[query_id] = ranked
    return results

Overwriting /kaggle/working/repo/research/offline_retrieval/bm25.py


### Bước 3 — Chạy BM25 trên pool `pan-pc11-simulated-v1`

Chạy BM25 (cả `fixed_window_bm25` và `structure_bm25`) trên pool mới, bằng đúng code đã khóa như lần chạy Test, rồi lưu kết quả về Drive `simulated_extra/`.

#### Bước 3a — Copy pool mới sang vùng ghi được, duyệt governance

- `/kaggle/input` chỉ đọc, nên copy pool `pan-pc11-simulated-v1` sang `/kaggle/working/data` trước.
- Runner từ chối chạy nếu `dataset-manifest.json` chưa ghi `governance = APPROVED`. PAN-PC-11 là corpus công khai cho nghiên cứu, nên tác giả tự duyệt và ghi tên, ngày duyệt vào manifest để truy vết (giống hai pool Test và Train).
- Pool này chỉ có một tập, nên chỉ có biến `SIM_DIR`.

Kết quả đúng: in `SIM_DIR = /kaggle/working/data/pan-pc11-simulated-v1` và `governance: APPROVED`.

In [6]:
import json, pathlib, os, shutil

REPO = "/kaggle/working/repo"
SRC = "/kaggle/input/datasets/yenkim1/pan-pc11-simulated-v1/pan-pc11-simulated-v1"
need = ['chunks.jsonl', 'dataset-manifest.json', 'documents.jsonl', 'qrels.jsonl', 'queries.jsonl']
assert all(os.path.exists(f"{SRC}/{f}") for f in need), "Thiếu file pool"

SIM_DIR = "/kaggle/working/data/pan-pc11-simulated-v1"
shutil.copytree(SRC, SIM_DIR)
print("SIM_DIR =", SIM_DIR, "|", sorted(os.listdir(SIM_DIR)))

mp = pathlib.Path(SIM_DIR) / "dataset-manifest.json"
manifest = json.loads(mp.read_text(encoding="utf-8"))
print("split name:", manifest["split"].get("name") or manifest["split"].get("split_name") or list(manifest["split"].keys())[:6])
manifest["governance"].update({
    "status": "APPROVED",
    "contains_pii": False,
    "retention": "Ephemeral Kaggle run; outputs giu lai lam bang chung thuc nghiem cho KLTN",
    "approver": "Vo Thi Kim Yen (tac gia KLTN, tu duyet du lieu cong khai PAN-PC-11 theo dieu khoan Zenodo)",
    "approved_at": "2026-11-25",
})
mp.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("governance:", manifest["governance"]["status"])

SIM_DIR = /kaggle/working/data/pan-pc11-simulated-v1 | ['chunks.jsonl', 'dataset-manifest.json', 'documents.jsonl', 'qrels.jsonl', 'queries.jsonl']
split name: simulated-v1
governance: APPROVED


#### Bước 3b — Chạy BM25 trên pool simulated

Gọi `python -m research.offline_retrieval run` với đúng config BM25 đã dùng cho Test (`pan-pc-11-bm25-chunking-pilot-v1.json`), chỉ đổi `experiment_id` thành `pan-pc-11-bm25-chunking-simulated-v1`. Chạy cả hai phương pháp: `fixed_window_bm25` (cửa sổ 256 từ, bước 128) và `structure_bm25` (đoạn tối đa 1.500 ký tự), lấy top 100 tài liệu cho mỗi truy vấn.

Pool có 179 parent, 280 truy vấn, 2.000 tài liệu, 399.148 chunk. Code là code đã khóa (bản BM25 nhanh bằng ma trận thưa), không đổi công thức so với lần chạy Test.

Kết quả ghi vào `/kaggle/working/results/simulated/bm25_chunking/` gồm `predictions.jsonl`, `metrics.json`, `run_manifest.json`.

Kết quả đúng: `rc=0`. Ô này im lặng đến khi xong.

In [7]:
import subprocess, time

OUT_ROOT = "/kaggle/working/results"
with open(f"{REPO}/research/configs/retrieval/pan-pc-11-bm25-chunking-pilot-v1.json") as f:
    cfg = dict(json.load(f))
cfg["experiment_id"] = "pan-pc-11-bm25-chunking-simulated-v1"
cfg_path = "/kaggle/working/sim_bm25_config.json"
json.dump(cfg, open(cfg_path, "w"), indent=2)

out_dir = f"{OUT_ROOT}/simulated/bm25_chunking"
pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)
cmd = ["python", "-m", "research.offline_retrieval", "run",
       "--dataset-manifest", f"{SIM_DIR}/dataset-manifest.json",
       "--config", cfg_path, "--output-dir", out_dir]
t0 = time.perf_counter()
r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode != 0:
    print("LỖI:", r.stderr[-3000:])
print(f"rc={r.returncode} | {(time.perf_counter()-t0)/60:.1f} phút")

/kaggle/working/results/simulated/bm25_chunking/run_manifest.json

rc=0 | 4.1 phút


#### Bước 3c — Đọc Recall và nén kết quả

In Recall@1, 5, 10, 20, 100 của hai phương pháp ở hai mức:
- `macro`: trung bình theo từng truy vấn (280 truy vấn);
- `parent_macro`: trung bình theo từng parent (179 parent). Số theo parent là số báo cáo chính, vì mỗi parent chỉ tính một lần.

Sau đó nén thư mục `results` thành `bm25_simulated_results.zip`. Tải file này về và lưu vào Drive `KLTN/simulated_extra/`.

In [8]:
import os, glob
base = "/kaggle/working/results"
print("results có không:", os.path.isdir(base))
for p in glob.glob(f"{base}/**", recursive=True)[:30]:
    print(p, "|", os.path.getsize(p) if os.path.isfile(p) else "thư mục")

results có không: True
/kaggle/working/results/ | thư mục
/kaggle/working/results/simulated | thư mục
/kaggle/working/results/simulated/bm25_chunking | thư mục
/kaggle/working/results/simulated/bm25_chunking/metrics.json | 351183
/kaggle/working/results/simulated/bm25_chunking/predictions.jsonl | 14390695
/kaggle/working/results/simulated/bm25_chunking/run_manifest.json | 104962


In [9]:
import json, shutil

out_dir = "/kaggle/working/results/simulated/bm25_chunking"
m = json.load(open(f"{out_dir}/metrics.json"))
for method, v in m.items():
    if isinstance(v, dict) and "parent_macro" in v:
        for scope in ("macro", "parent_macro"):
            row = {k: round(v[scope][k]["recall"], 4) for k in ["1", "5", "10", "20", "100"]}
            print(f"{method:20s} {scope:13s} recall@k = {row}")

shutil.make_archive("/kaggle/working/bm25_simulated_results", "zip", "/kaggle/working/results")
print("Đã nén: /kaggle/working/bm25_simulated_results.zip")

fixed_window_bm25    macro         recall@k = {'1': 0.95, '5': 0.975, '10': 0.975, '20': 0.9786, '100': 1.0}
fixed_window_bm25    parent_macro  recall@k = {'1': 0.933, '5': 0.9683, '10': 0.9683, '20': 0.9739, '100': 1.0}
structure_bm25       macro         recall@k = {'1': 0.9571, '5': 0.9679, '10': 0.9714, '20': 0.9821, '100': 0.9964}
structure_bm25       parent_macro  recall@k = {'1': 0.9404, '5': 0.9572, '10': 0.9628, '20': 0.9795, '100': 0.9944}
Đã nén: /kaggle/working/bm25_simulated_results.zip


### Kết quả BM25 trên pool `pan-pc11-simulated-v1`

Pool: 179 parent (simulated), 280 truy vấn, 2.000 tài liệu (161 nguồn dương, 1.839 distractor), 399.148 chunk. Code BM25 là code đã khóa (bản nhanh bằng ma trận thưa), cùng cấu hình với lần chạy Test. Thời gian chạy: 4,1 phút trên CPU.

| Phương pháp | Mức | R@1 | R@5 | R@10 | R@20 | R@100 |
|---|---|---:|---:|---:|---:|---:|
| fixed_window_bm25 | parent | 0,933 | 0,968 | 0,968 | 0,974 | 1,000 |
| structure_bm25 | parent | 0,940 | 0,957 | 0,963 | 0,980 | 0,994 |
| fixed_window_bm25 | truy vấn | 0,950 | 0,975 | 0,975 | 0,979 | 1,000 |
| structure_bm25 | truy vấn | 0,957 | 0,968 | 0,971 | 0,982 | 0,996 |

**Cách đọc:**
- Số mức **parent** (179 parent) là số báo cáo chính, vì mỗi parent chỉ tính một lần. Số mức truy vấn (280) chỉ để tham khảo.
- BM25 gần đạt trần trên tập này: R@100 mức parent là 1,000 (cửa sổ cố định) và 0,994 (structure), R@1 khoảng 0,93 đến 0,94. Nhóm simulated của Test cũng đạt R@100 bằng 1,0. Vì vậy khả năng phân biệt các hệ thống nằm chủ yếu ở R@1 và R@5.
- Chênh lệch giữa hai cách chunking rất nhỏ và không cùng chiều ở mọi k (cửa sổ cố định hơn ở R@5, R@10, R@100; structure hơn ở R@1, R@20). Với 179 parent, một parent làm đổi khoảng 0,6 điểm phần trăm, nên chưa kết luận về hai cách chunking từ bảng này.
- Tập này đã loại trừ toàn bộ parent, tài liệu nghi vấn và tài liệu nguồn của 5 split trước, nên độc lập với Train, Val và Test.

Kết quả ghi tại `/kaggle/working/results/simulated/bm25_chunking/` (`predictions.jsonl`, `metrics.json`, `run_manifest.json`), nén thành `bm25_simulated_results.zip`.